# 03 Synthetic Data Evaluation

This notebook evaluates motif discovery algorithms on synthetic DNA datasets.

Experiments test how performance is affected by:

- mismatch level
- motif length
- number of sequences
- sequence length

Performance is measured using:

- Hamming distance between discovered motif and planted motif
- runtime

MEME is included as an external baseline.

In [ ]:
import sys
import os
import time
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

project_root = os.path.abspath("..")
sys.path.append(project_root)

from src.data_utils import generate_dataset, hamming
from src.random_search import random_search
from src.rmhc import rmhc
from src.ga import genetic_algorithm
from src.aco import aco_motif_search
from src.simulated_annealing import simulated_annealing
from src.fasta_utils import save_fasta

### Run algorithms

In [ ]:
def run_custom_algorithms(sequences, k):
    results = []

    # Random Search & record discovered motif + runtime 
    start = time.time()
    _, score, consensus = random_search(sequences, k, iterations=2000)
    runtime = time.time() - start
    results.append(("Random Search", consensus, runtime))

    # RMHC
    start = time.time()
    _, score, consensus = rmhc(sequences, k, iterations=2000)
    runtime = time.time() - start
    results.append(("RMHC", consensus, runtime))

    # GA
    start = time.time()
    _, score, consensus = genetic_algorithm(
        sequences,
        k,
        population_size=30,
        generations=200
    )
    runtime = time.time() - start
    results.append(("GA", consensus, runtime))

    # ACO
    start = time.time()
    _, score, consensus = aco_motif_search(
        sequences,
        k,
        num_ants=20,
        iterations=100
    )
    runtime = time.time() - start
    results.append(("ACO", consensus, runtime))

        # Simulated Annealing
    start = time.time()
    _, score, consensus = simulated_annealing(
        sequences,
        k,
        iterations=2000,
        initial_temperature=10.0,
        cooling_rate=0.995
    )
    runtime = time.time() - start
    results.append(("SA", consensus, runtime))

    return results

## Run MEME
I am running MEME using WSL. Exuction time is also measured

In [ ]:
def windows_to_wsl_path(path):
    # Convert a Windows path into WSL format so MEME can access the file
    path = os.path.abspath(path)
    drive = path[0].lower()
    rest = path[2:].replace("\\", "/")
    return f"/mnt/{drive}{rest}"


def run_meme(fasta_file, k, output_dir):
    # Convert input and output paths from Windows format to WSL format
    wsl_fasta_file = windows_to_wsl_path(fasta_file)
    wsl_output_dir = windows_to_wsl_path(output_dir)
 # Build MEME command with fixed motif width k and DNA mode enabled
    cmd = [
        "wsl",
        os.environ.get("MEME_BIN", "meme"),
        wsl_fasta_file,
        "-dna",
        "-mod", "oops",
        "-nmotifs", "1",
        "-minw", str(k),
        "-maxw", str(k),
        "-oc", wsl_output_dir
    ]
    # Execute MEME and measure runtime

    start = time.time()
    subprocess.run(cmd, check=True)
    runtime = time.time() - start
    return runtime

## Convert MEME output to Consensus 
- Convert MEME's matrix into a consensus sequence for comparison

In [ ]:
def pwm_to_consensus(pwm):
    # Convert a position weight/probability matrix into a consensus string
    # by selecting the highest-probability base at each position
    letters = ["A", "C", "G", "T"]
    consensus = ""

    for row in pwm:
        max_index = row.index(max(row))
        consensus += letters[max_index]

    return consensus


def extract_meme_consensus(meme_txt_file):
    pwm = []
    reading_matrix = False

    with open(meme_txt_file, "r") as f:
        for line in f:
            line = line.strip()
                # Detect the start of MEME's letter-probability matrix
            if "letter-probability matrix" in line:
                reading_matrix = True
                continue

            if reading_matrix:
                if not line or line.startswith("-"): # Stop reading when matrix section ends
                    break

                parts = line.split()

                # matrix rows should have 4 probability values: A C G T
                if len(parts) >= 4:
                    row = [float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])]
                    pwm.append(row)

    if not pwm:
        raise ValueError(f"Could not extract PWM from {meme_txt_file}")
        
    # Convert extracted matrix into a single consensus motif

    return pwm_to_consensus(pwm)

## Test MEME output 
- check that meme results are correctly read and converted 

In [ ]:
# Test whether the MEME parser can correctly extract a consensus motif

test_meme_file = "../results/synthetic/meme/mismatch_0_run_0/meme.txt"
test_consensus = extract_meme_consensus(test_meme_file)

print("Extracted MEME consensus:", test_consensus)

In [ ]:
# Check that MEME is installed and accessible through WSL

subprocess.run(["wsl", os.environ.get("MEME_BIN", "meme"), "-version"], check=True)

In [ ]:
#create folders to store fastsa files, results, plots and tables
os.makedirs("../data/synthetic/fasta", exist_ok=True)
os.makedirs("../results/synthetic/meme", exist_ok=True)
os.makedirs("../results/synthetic/plots", exist_ok=True)
os.makedirs("../results/synthetic/tables", exist_ok=True)

### Experiment 1: Mismatches 

In [ ]:
# Experiment 1: test how motif recovery changes as mismatch level increases
motif = "ACGTAC"
num_sequences = 5
sequence_length = 30
runs = 5  

# Mismatch levels range from perfect matches to maximum mutation of the motif

mismatch_values = list(range(0, len(motif) + 1))
results = []

for mismatches in mismatch_values:
    print("Mismatch level:", mismatches)

    for run in range(runs):

        # Generate dataset with planted motif and select mismatch levels
        sequences, true_positions, planted_copies = generate_dataset(
            motif,
            num_sequences,
            sequence_length,
            mismatches
        )

        k = len(motif)

        # Save data in FASTA for MEME
        fasta_file = f"../data/synthetic/fasta/mismatch_{mismatches}_run_{run}.fasta"
        save_fasta(sequences, fasta_file)

        # Run custom algorithms
        algo_results = run_custom_algorithms(sequences, k)

        for name, consensus, runtime in algo_results:
            distance = hamming(consensus, motif)

            results.append({
                "method": name,
                "mismatch": mismatches,
                "run": run,
                "hamming_distance": distance,
                "runtime": runtime
            })

        # Run MEME
        output_dir = f"../results/synthetic/meme/mismatch_{mismatches}_run_{run}"
        meme_runtime = run_meme(fasta_file, k, output_dir)
        
        # Extract MEME's discovered motif and compare it with the planted motif

        meme_txt_file = os.path.join(output_dir, "meme.txt")
        meme_consensus = extract_meme_consensus(meme_txt_file)
        meme_distance = hamming(meme_consensus, motif)

        results.append({
            "method": "MEME",
            "mismatch": mismatches,
            "run": run,
            "hamming_distance": meme_distance,
            "runtime": meme_runtime
        })

## Results summary
The table below shows the average score and average runtime across repeated runs for each mismatch level and algorithm.

In [ ]:
# Store raw results from all runs in a DataFrame for analysis

df = pd.DataFrame(results)
print(df)

In [ ]:
# Compute mean accuracy and runtime for each method at each mismatch level

summary = df.groupby(["method", "mismatch"], as_index=False).agg(
    avg_hamming=("hamming_distance", "mean"),
    avg_runtime=("runtime", "mean")
)

summary

In [ ]:
# Save mismatch experiment summary table for later reporting

summary.to_csv("../results/synthetic/tables/mismatch_summary.csv", index=False)
print("Saved to ../results/synthetic/tables/mismatch_summary.csv")

In [ ]:
# Plot average Hamming distance against mismatch level for all methods

plt.figure(figsize=(8, 5))

for method in summary["method"].unique():
    subset = summary[summary["method"] == method]
    plt.plot(subset["mismatch"], subset["avg_hamming"], marker="o", label=method)

plt.xlabel("Number of mismatches")
plt.ylabel("Average Hamming distance")
plt.title("Motif recovery accuracy vs mismatch level")
plt.legend()
plt.grid(True)
plt.savefig("../results/synthetic/plots/mismatch_hamming_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Plot average runtime against mismatch level for all methods
plt.figure(figsize=(8, 5))

for method in summary["method"].unique():
    subset = summary[summary["method"] == method]
    plt.plot(subset["mismatch"], subset["avg_runtime"], marker="o", label=method)

plt.xlabel("Number of mismatches")
plt.ylabel("Average runtime (seconds)")
plt.title("Runtime vs mismatch level")
plt.legend()
plt.grid(True)
plt.savefig("../results/synthetic/plots/mismatch_runtime_plot.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Experiment 2: effect of motif length

This experiment tests how algorithm performance changes as the planted motif length increases.

For this experiment, the background sequence length is increased to 50 so that longer motifs still sit naturally within the synthetic DNA sequences.

In [ ]:
# Experiment 2: test how motif recovery changes as motif length increases

motif_lengths = [4, 6, 8, 10, 12]
num_sequences = 5
sequence_length = 50
max_mismatches = 1
runs = 5

motif_length_results = []

In [ ]:
for motif_length in motif_lengths:
    print("Motif length:", motif_length)

    # Build a motif of the required length
    motif = "ACGT" * ((motif_length // 4) + 1)
    motif = motif[:motif_length]

    for run in range(runs):
        # Generate dataset
        sequences, true_positions, planted_copies = generate_dataset(
            motif,
            num_sequences,
            sequence_length,
            max_mismatches
        )

        k = len(motif)

        # Save FASTA for MEME
        fasta_file = f"../data/synthetic/fasta/motif_length_{motif_length}_run_{run}.fasta"
        save_fasta(sequences, fasta_file)

        # Run custom algorithms
        algo_results = run_custom_algorithms(sequences, k)

        for name, consensus, runtime in algo_results:
            distance = hamming(consensus, motif)

            motif_length_results.append({
                "method": name,
                "motif_length": motif_length,
                "run": run,
                "hamming_distance": distance,
                "runtime": runtime
            })

        # Run MEME
        output_dir = f"../results/synthetic/meme/motif_length_{motif_length}_run_{run}"
        meme_runtime = run_meme(fasta_file, k, output_dir)

        meme_txt_file = os.path.join(output_dir, "meme.txt")
        meme_consensus = extract_meme_consensus(meme_txt_file)
        meme_distance = hamming(meme_consensus, motif)

        motif_length_results.append({
            "method": "MEME",
            "motif_length": motif_length,
            "run": run,
            "hamming_distance": meme_distance,
            "runtime": meme_runtime
        })

In [ ]:
# Store motif length experiment results in a DataFrame

df_motif_length = pd.DataFrame(motif_length_results)
print(df_motif_length)

## Motif length summary table
The table below shows the average score and average runtime across repeated runs for each motif length and algorithm.

In [ ]:
# Compute mean accuracy and runtime for each method at each motif length

motif_length_summary = df_motif_length.groupby(["method", "motif_length"], as_index=False).agg(
    avg_hamming=("hamming_distance", "mean"),
    avg_runtime=("runtime", "mean")
)

motif_length_summary

In [ ]:
# Save motif length experiment summary table

motif_length_summary.to_csv("../results/synthetic/tables/motif_length_summary.csv", index=False)
print("Saved to ../results/synthetic/tables/motif_length_summary.csv")

In [ ]:
# Plot average Hamming distance against motif length for all methods

plt.figure(figsize=(8, 5))

for method in motif_length_summary["method"].unique():
    subset = motif_length_summary[motif_length_summary["method"] == method]
    plt.plot(subset["motif_length"], subset["avg_hamming"], marker="o", label=method)

plt.xlabel("Motif length")
plt.ylabel("Average Hamming distance")
plt.title("Motif recovery accuracy vs motif length")
plt.legend()
plt.grid(True)
plt.savefig("../results/synthetic/plots/motif_length_hamming_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Plot average runtime against motif length for all methods

plt.figure(figsize=(8, 5))

for method in motif_length_summary["method"].unique():
    subset = motif_length_summary[motif_length_summary["method"] == method]
    plt.plot(subset["motif_length"], subset["avg_runtime"], marker="o", label=method)

plt.xlabel("Motif length")
plt.ylabel("Average runtime (seconds)")
plt.title("Runtime vs motif length")
plt.legend()
plt.grid(True)
plt.savefig("../results/synthetic/plots/motif_length_runtime_plot.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Experiment 3: effect of number of sequences
This experiment tests how algorithm performance changes as the number of input sequences increases.

The planted motif, sequence length, and mismatch level are kept fixed.

In [ ]:
sequence_counts = [3, 5, 7, 10]
motif = "ACGTAC"
sequence_length = 30
max_mismatches = 1
runs = 5
# Store results for all methods across all sequence count 
sequence_count_results = []

In [ ]:
for num_sequences in sequence_counts:
    print("Number of sequences:", num_sequences)

    for run in range(runs):
        # Generate dataset
        sequences, true_positions, planted_copies = generate_dataset(
            motif,
            num_sequences,
            sequence_length,
            max_mismatches
        )

        k = len(motif)

        # Save FASTA for MEME
        fasta_file = f"../data/synthetic/fasta/sequence_count_{num_sequences}_run_{run}.fasta"
        save_fasta(sequences, fasta_file)

        # Run custom algorithms
        algo_results = run_custom_algorithms(sequences, k)

        for name, consensus, runtime in algo_results:
            distance = hamming(consensus, motif)

            sequence_count_results.append({
                "method": name,
                "num_sequences": num_sequences,
                "run": run,
                "hamming_distance": distance,
                "runtime": runtime
            })

        # Run MEME
        output_dir = f"../results/synthetic/meme/sequence_count_{num_sequences}_run_{run}"
        meme_runtime = run_meme(fasta_file, k, output_dir)

        meme_txt_file = os.path.join(output_dir, "meme.txt")
        meme_consensus = extract_meme_consensus(meme_txt_file)
        meme_distance = hamming(meme_consensus, motif)

        sequence_count_results.append({
            "method": "MEME",
            "num_sequences": num_sequences,
            "run": run,
            "hamming_distance": meme_distance,
            "runtime": meme_runtime
        })

In [ ]:
# Convert raw results into a DataFrame for analysis
df_sequence_count = pd.DataFrame(sequence_count_results)
print(df_sequence_count)

In [ ]:
# Compute average accuracy and runtime for each method at each sequence count
sequence_count_summary = df_sequence_count.groupby(["method", "num_sequences"], as_index=False).agg(
    avg_hamming=("hamming_distance", "mean"),
    avg_runtime=("runtime", "mean")
)

sequence_count_summary

In [ ]:
# Save sequence count experiment summary table
sequence_count_summary.to_csv("../results/synthetic/tables/sequence_count_summary.csv", index=False)
print("Saved to ../results/synthetic/tables/sequence_count_summary.csv")

In [ ]:
# Plot average Hamming distance against number of sequences for all methods
plt.figure(figsize=(8, 5))

for method in sequence_count_summary["method"].unique():
    subset = sequence_count_summary[sequence_count_summary["method"] == method]
    plt.plot(subset["num_sequences"], subset["avg_hamming"], marker="o", label=method)

plt.xlabel("Number of sequences")
plt.ylabel("Average Hamming distance")
plt.title("Motif recovery accuracy vs number of sequences")
plt.legend()
plt.grid(True)
plt.savefig("../results/synthetic/plots/sequence_count_hamming_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Plot average runtime against number of sequences for all methods
plt.figure(figsize=(8, 5))

for method in sequence_count_summary["method"].unique():
    subset = sequence_count_summary[sequence_count_summary["method"] == method]
    plt.plot(subset["num_sequences"], subset["avg_runtime"], marker="o", label=method)

plt.xlabel("Number of sequences")
plt.ylabel("Average runtime (seconds)")
plt.title("Runtime vs number of sequences")
plt.legend()
plt.grid(True)
plt.savefig("../results/synthetic/plots/sequence_count_runtime_plot.png", dpi=300, bbox_inches="tight")
plt.show()

## Experiment 4 — Effect of DNA sequence length

In [ ]:
sequence_lengths = [20, 30, 40, 50, 75, 100]
motif = "ACGTAC"
num_sequences = 5
max_mismatches = 1
runs = 5

sequence_length_results = []

In [ ]:
for sequence_length in sequence_lengths:
    print("Sequence length:", sequence_length)

    for run in range(runs):
        # Generate dataset
        sequences, true_positions, planted_copies = generate_dataset(
            motif,
            num_sequences,
            sequence_length,
            max_mismatches
        )

        k = len(motif)

        # Save FASTA for MEME
        fasta_file = f"../data/synthetic/fasta/sequence_length_{sequence_length}_run_{run}.fasta"
        save_fasta(sequences, fasta_file)

        # Run custom algorithms
        algo_results = run_custom_algorithms(sequences, k)

        for name, consensus, runtime in algo_results:
            distance = hamming(consensus, motif)

            sequence_length_results.append({
                "method": name,
                "sequence_length": sequence_length,
                "run": run,
                "hamming_distance": distance,
                "runtime": runtime
            })

        # Run MEME
        output_dir = f"../results/synthetic/meme/sequence_length_{sequence_length}_run_{run}"
        meme_runtime = run_meme(fasta_file, k, output_dir)

        meme_txt_file = os.path.join(output_dir, "meme.txt")
        meme_consensus = extract_meme_consensus(meme_txt_file)
        meme_distance = hamming(meme_consensus, motif)

        sequence_length_results.append({
            "method": "MEME",
            "sequence_length": sequence_length,
            "run": run,
            "hamming_distance": meme_distance,
            "runtime": meme_runtime
        })

In [ ]:
# Convert raw sequence length experiment results into a DataFrame
df_sequence_length = pd.DataFrame(sequence_length_results)
df_sequence_length

In [ ]:
# Compute average accuracy and runtime for each method at each sequence length
sequence_length_summary = df_sequence_length.groupby(["method", "sequence_length"], as_index=False).agg(
    avg_hamming=("hamming_distance", "mean"),
    avg_runtime=("runtime", "mean")
)

sequence_length_summary

In [ ]:
# Save sequence length experiment summary table
sequence_length_summary.to_csv("../results/synthetic/tables/sequence_length_summary.csv", index=False)
print("Saved to ../results/synthetic/tables/sequence_length_summary.csv")

In [ ]:
# Plot average Hamming distance against sequence length for all methods
plt.figure(figsize=(8, 5))

for method in sequence_length_summary["method"].unique():
    subset = sequence_length_summary[sequence_length_summary["method"] == method]
    plt.plot(subset["sequence_length"], subset["avg_hamming"], marker="o", label=method)

plt.xlabel("Sequence length")
plt.ylabel("Average Hamming distance")
plt.title("Motif recovery accuracy vs sequence length")
plt.legend()
plt.grid(True)
plt.savefig("../results/synthetic/plots/sequence_length_hamming_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Plot average runtime against sequence length for all methods
plt.figure(figsize=(8, 5))

for method in sequence_length_summary["method"].unique():
    subset = sequence_length_summary[sequence_length_summary["method"] == method]
    plt.plot(subset["sequence_length"], subset["avg_runtime"], marker="o", label=method)

plt.xlabel("Sequence length")
plt.ylabel("Average runtime (seconds)")
plt.title("Runtime vs sequence length")
plt.legend()
plt.grid(True)
plt.savefig("../results/synthetic/plots/sequence_length_runtime_plot.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Synthetic evaluation summary
This notebook evaluated the implemented motif-discovery algorithms on synthetic datasets under four controlled conditions:

- increasing mismatch level
- increasing motif length
- increasing number of sequences

For each experiment, repeated runs were used to reduce the effect of randomness, and average score and runtime were recorded.

These results will be used in the final report to compare the strengths and limitations of the different algorithms under controlled synthetic conditions.